# 07 · Summary statistics

From τ of several lines, `DeltaFieldBuilder` forms, per skewer, the overflux δ_j = F_j/⟨F_j⟩ − 1 of every line, the
products δ_A of every line subset and the total δ_tot = ∏F_j/⟨∏F_j⟩ − 1, which decomposes **exactly** into the
subset terms (review notes, Sec. Overflux). `OverfluxStatistics1D` gives P1D and ξ1D (an exact discrete
Fourier pair; P_ab = (dv/N) F_a F_b^*, the review notes) of the total, of every line pair and of every order
|A| + |B|; `OverfluxStatistics3D` does the same on a grid of skewers. Runtime: ~30 s.

**Requires:** the optical depth of several lines (06) and their offsets Δv_j (01).  
**Independent of:** how the τ was computed (core or simulation).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('overflux', 'stats',)); plt.show()

## 1 · From the integrator's output to δ_F, step by step

The integrator returns one τ per line, each in its **own** line-centred frame. Four steps lead to the overflux:
(a) F_j = e^{−τ_j} per line; (b) every line moved onto the reference (Lyα) frame by `wrap_to_ref_line`, a shift by
−Δv_j that wraps around the grid; (c) the total F = ∏_j F_j; (d) the mean flux ⟨F⟩ and δ_F = F/⟨F⟩ − 1, per line and for
the total (`DeltaFieldBuilder`). The toy box of notebook 00 (50 cMpc, z = 2.2, one GRF for H I and Si III), integrated
with the default (direct) sum on a grid of exactly one box length in velocity; how the integral treats the box edges
is discussed in notebook 06, section 8.

In [ ]:
from skewerjax.core import (LineBundle, Cosmology, SkewerBundle, grid_skewer_points, phys, Fields, ScalarField, VectorField,
                            PowerLawGRF, GaussianDensity, LogRamp, VelocityRamp, Constant, compute_optical_depth, wrap_to_ref_line)
from skewerjax.utils.plotting import plot_raw_to_overflux
lines2 = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
Lb, z = 50.0, 2.2
pts = grid_skewer_points(Lb, 8, 8193, direction="y").reshape(64, 8193, 3).astype(jnp.float64)   # first and last point: the same place
sbq = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z), x0=Lb)
a_d = 1 / (1 + z)
P = float(phys.c_kms * jnp.log(a_d / sbq.scale_factors[0, 0, 0]))              # the box in velocity: one period [km/s]
N_v = 2048
dvq = P / N_v
vq = (np.arange(N_v) + 0.5) * dvq                                                # one period, N_v dv = P exactly
grf_q = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, L=(Lb,) * 3, N=(64,) * 3)
key_q = jax.random.PRNGKey(0)
dens_q = {"HI": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11, contrast=1.2),
          "SiIII": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11 / 250, contrast=1.2)}
toy_sp = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, Lb)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lb)),
                    "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
toy = Fields({sp: toy_sp(d) for sp, d in dens_q.items()})
tau_own = compute_optical_depth(sbq, vq, lines2, toy, a_dagger=a_d)                         # the default: the direct sum over the points
tau_ref = wrap_to_ref_line(tau_own, vq, lines2)
dV = float(lines2.delta_v[1])
print(f"P = {P:.1f} km/s, dv = {dvq:.3f} km/s, Delta v(Si III) = {dV:.1f} km/s = {dV / P:+.3f} P | <F>: Lya {np.exp(-np.asarray(tau_ref[0])).mean():.4f}, "
      f"Si III {np.exp(-np.asarray(tau_ref[1])).mean():.4f}, total {np.exp(-np.asarray(tau_ref)).prod(0).mean():.4f}")
fig, axs = plot_raw_to_overflux(vq, tau_own, tau_ref, ["HI-LyA", "SiIII-1206"], skewer=0, period=P)
fig.suptitle("From the integrator's raw output to the overflux (skewer 0 of 64)", fontsize=22); plt.show()

In [ ]:
from skewerjax.core import PowerLawGRF, DeltaFieldBuilder
from skewerjax.core.stats import OverfluxStatistics1D, OverfluxStatistics3D
from skewerjax.utils.plotting import plot_orders_superimposed, plot_pk_recovery, hybrid_k_edges, logbin
N_sk, N_v, Lv = 400, 1000, 1000.0                         # skewers, pixels, velocity length [km/s]
dv, shift = Lv / N_v, 100                                  # the second line sits 100 km/s away
grf = PowerLawGRF(1.0, 2 * np.pi / 100, 0.5, L=Lv, N=N_v)
grf = PowerLawGRF(1.0 / float(grf.variance()), 2 * np.pi / 100, 0.5, L=Lv, N=N_v)     # sigma_delta = 1
delta = jax.vmap(grf.sample)(jax.random.split(jax.random.PRNGKey(7), N_sk))
tau = jnp.stack([jnp.exp(delta), 0.5 * jnp.exp(jnp.roll(delta, shift, axis=-1))])        # line 2: same gas, shifted, weaker
ov = DeltaFieldBuilder(tau, labels=["1", "2"], keep_fields=True).build()
print("subset keys:", ov.subset_keys, "| C_s of the first skewers:", np.round(np.asarray(ov.C_per_sample[:3, 0]), 4))
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
vv = np.arange(N_v) * dv
for j, lab in enumerate(ov.labels):
    axs[0].plot(vv, ov.fields[j, 0], color=list(TOL.values())[j], label=f"$F_{lab}$"); axs[1].plot(vv, ov.delta_terms[lab][0], color=list(TOL.values())[j])
axs[1].plot(vv, ov.total[0], color=FG, lw=1.2, label=r"$\delta_{\rm tot}$")
axs[0].set(ylabel="F", xlim=(0, 300)); axs[1].set(xlabel="v [km/s]", ylabel=r"$\delta$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle("Two lines of the same gas: the second is a shifted, weaker copy", fontsize=22); plt.show()

## 2 · P1D and ξ1D by order

Order n collects the terms (A, B) with |A| + |B| = n (order 0 is the normalisation term 𝒞). Their sum is the total to machine precision.

In [ ]:
st = OverfluxStatistics1D(ov, dv).compute_p1d().compute_xi1d()
fig, axs = plot_orders_superimposed(st, "P", delta_v_list=[0.0, shift * dv], figsize=(13, 9))
axs[0].legend(fontsize=15, ncol=2, loc="lower left")
fig.suptitle(r"P1D: orders 2, 3, 4 add up to the total; the cross terms peak at $k = 2\pi n/\Delta v$ (dashed)", fontsize=20)
fig, axs = plot_orders_superimposed(st, "xi", delta_v_list=[0.0, shift * dv], max_lag=300.0, figsize=(12, 8))
fig.suptitle(r"$\xi_{\rm 1D}$: the cross term peaks at the line separation (dashed)", fontsize=20); plt.show()

**The terms of two lines.** With two lines, F = F̄_1F̄_2(1 + δ_1)(1 + δ_2), and the total overflux is
δ = [δ_1 + δ_2 + δ_1δ_2 − C_12]/(1 + C_12), with C_12 = ⟨δ_1δ_2⟩ the normalisation term (𝒞 below). Every term of
ξ1D (real space) or P1D (Fourier space) is a pair of subsets A, B of {1, 2}; its order is |A| + |B|:

| A \ B | 𝒞 (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| **𝒞** | +𝒞𝒞 | 0 | 0 | −C_12 𝒞 |
| **1** | 0 | ξ_11 · P_11 | ξ_21 · P_21 | ξ_(12)1 · P_(12)1 |
| **2** | 0 | ξ_12 · P_12 | ξ_22 · P_22 | ξ_(12)2 · P_(12)2 |
| **12** | −𝒞 C_12 | ξ_1(12) · P_1(12) | ξ_2(12) · P_2(12) | ξ_(12)(12) · P_(12)(12) |

After the ensemble average,

$$\xi = \frac{-C_{12}\mathcal{C} + \xi_{11} + \xi_{22} + \xi_{12,\rm sym} + \xi_{1(12),\rm sym} + \xi_{2(12),\rm sym} + \xi_{(12)(12)}}{(1 + C_{12})^2},\qquad
P(k) = \frac{P_{11} + P_{22} + P_{12,\rm sym} + P_{1(12),\rm sym} + P_{2(12),\rm sym} + P_{(12)(12)} - 2\pi C_{12}^2\,\delta_D(k)}{(1 + C_{12})^2},$$

where "sym" adds the pair and its transpose (1/2 with 2/1, 1/12 with 12/1, 2/12 with 12/2). In Fourier space the constant
pieces only touch the zero mode. For a second line that is a shifted copy of the first, δ_2(v) ∝ δ_1(v − Δv), the cross
term is P_12,sym ∝ 2 cos(kΔv) P_11: it oscillates with period 2π/Δv in k (the dashed lines above), the Fourier image of
the ξ1D peak at Δv.

### Each order, term by term

The same decomposition one level down: every term (A, B) of order |A| + |B| = n, with (A, B) and (B, A) drawn
together as A–B (their sum, which is real). 𝒞 is the order-0 normalisation term. Order 2 holds the Gaussian-like
products (1–1, 1–2, 2–2) and the normalisation cross term 𝒞–12; orders 3 and 4 are the non-linear corrections of
the product of the two fluxes.

In [ ]:
from skewerjax.utils.plotting import plot_terms_by_order
fig, axs = plot_terms_by_order(st, "P")
fig.suptitle("P1D: every order split into its terms; the terms add up to the order (white)", fontsize=20); plt.show()
fig, axs = plot_terms_by_order(st, "xi", max_lag=300.0)
for ax in axs:
    ax.axvline(shift * dv, color=FG, ls=":", lw=1.2)
fig.suptitle(r"$\xi_{\rm 1D}$ by order and term: the cross terms 1–2 carry the line separation (dotted)", fontsize=20); plt.show()
P_terms = {n: sum(np.asarray(st.Pk_total_from_terms_contributions[p]) for p in st.term_pairs if st.pair_orders[p] == n) for n in st.Pk_total_by_order}
print("max |sum of the terms - order| / max|total| per order:",
      {n: f"{np.max(np.abs(P_terms[n] - np.asarray(st.Pk_total_by_order[n]))) / np.max(np.abs(np.asarray(st.Pk_total))):.1e}" for n in P_terms})

## 3 · Recovering the input spectrum: a systematic study

One GRF with a known input P_in(k) sets the H I and Si III densities of a 50 cMpc box (z = 2.2, T = 1.5 × 10⁴ K, no
peculiar velocities). Every signal is measured with the same estimator and compared with the input:
- **(a) in 3D**, on the grid: the overdensities δ_n = n/⟨n⟩ − 1 of H I and Si III, and the overfluxes δ_F of Lyα and
  Si III (the τ of a 32 × 32 grid of skewers along every other grid column, 8192 points each);
- **(b) in 1D**, along the same skewers: the overdensities of H I, Si III and of their sum, and the overfluxes of Lyα,
  Si III and of the total flux. The expected 1D curve is the discrete projection of the input,
  P1D(k_∥) = L_⊥⁻² Σ_{k_⊥} P_in(√(k_∥² + k_⊥²)) (`plotting.grf_p1d_along_axis`), the exact 1D spectrum of grid lines
  sampled at the nodes;
- **(c) the 3D analysis on the grid of skewers**: P3D(k_⊥, k_∥) and ξ3D(r_⊥, π) of the Lyα overflux.

`GaussianDensity` is used in its linear regime (a large `contrast`): log₁₀ n = log₁₀ n̄ + A tanh(δ/contrast) ≈ log₁₀ n̄ +
(A/contrast) δ, so δ_n ≈ b δ with the linear bias b = ln 10 · A/contrast (A: half-width of log n in dex). Each
overdensity is divided by its b and must recover P_in. The overfluxes are biased and smoothed (thermal widths,
saturation): δ_F ≈ b_F δ at low k, with b_F² measured at the largest scales against the H I overdensity of the same
realisation (so that the cosmic variance of one box cancels), and they fall below the input at high k.

In [ ]:
from types import SimpleNamespace
from skewerjax.utils.plotting import binned_power, grf_p1d_along_axis, plot_p1d_input_vs_measured
N3, L3 = 64, 50.0
grf3 = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, L=(L3,) * 3, N=(N3,) * 3)
key3 = jax.random.PRNGKey(11)
A_dex, contrast = {"HI": 1.0, "SiIII": 0.5}, 8.0                                  # half-widths of log n [dex]; linear regime
n_bar = {"HI": 2.5e-12, "SiIII": 2.5e-12 / 50}
dens3 = {sp: GaussianDensity(key3, grf3, n0_min=n_bar[sp] / 10 ** A_dex[sp], n0_max=n_bar[sp] * 10 ** A_dex[sp], contrast=contrast)
         for sp in ("HI", "SiIII")}
b_lin = {sp: np.log(10) * A_dex[sp] / contrast for sp in dens3}                  # delta_n ~ b delta (delta in units of sigma)
sig3 = float(jnp.sqrt(grf3.variance()))
# the densities at every node of the grid (events (a, x, y, z)), the line of sight (y) last
gx = np.arange(N3) * (L3 / N3)
X, Z, Y = np.meshgrid(gx, gx, gx, indexing="ij")
ev3 = jnp.asarray(np.stack([np.ones(X.size), X.ravel(), Y.ravel(), Z.ravel()], -1))
n3 = np.stack([np.asarray(dens3[sp](ev3)).reshape(N3, N3, N3) for sp in ("HI", "SiIII")])     # (2, x, z, y)
# the skewers: every other grid column (x_i, z_j), along y, 8192 points (Si III resolved); the first point at y = 0
Ns, n_sub = N3 // 2, 128
ys = np.arange(N3 * n_sub) * (L3 / (N3 * n_sub))
P3 = np.stack(np.meshgrid(gx[::2], gx[::2], indexing="ij"), -1).reshape(-1, 2)
pts3 = jnp.asarray(np.stack([np.repeat(P3[:, :1], ys.size, 1), np.broadcast_to(ys, (P3.shape[0], ys.size)), np.repeat(P3[:, 1:], ys.size, 1)], -1))
sb3d = SkewerBundle(spatial_points=pts3, cosmologies=eds, a0=1 / (1 + z), x0=L3)
sp3 = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, L3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, L3)),
                 "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
toy3 = Fields({sp: sp3(d) for sp, d in dens3.items()})
a3d = np.asarray(sb3d.scale_factors[0])
P3v = float(phys.c_kms * np.log(a3d.max() / a3d.min())) * ys.size / (ys.size - 1)     # the box in velocity [km/s]
N_v3 = N3 * 4
v3 = (np.arange(N_v3) + 0.5) * P3v / N_v3 + float(phys.c_kms * np.log((1 / (1 + z)) / a3d.max()))
tau3 = wrap_to_ref_line(compute_optical_depth(sb3d, v3, lines2, toy3, a_dagger=1 / (1 + z), resolution_tolerance=0.3), v3, lines2)   # Si III: dv_c / b = 0.26
F3 = np.exp(-np.asarray(tau3))
u3 = P3v / L3                                                                          # km/s per cMpc (Hubble flow of the box)
print(f"{P3.shape[0]} skewers x {ys.size} points; box {P3v:.0f} km/s ({u3:.1f} km/s per cMpc); <F>: Lya {F3[0].mean():.3f}, "
      f"Si III {F3[1].mean():.4f}, total {F3.prod(0).mean():.3f}; linear biases b = {b_lin}")

In [ ]:
# (a) 3D: the fields on the grid (x, z, y), the line of sight last
grid3 = lambda f: f.reshape(Ns, Ns, -1)                                               # the skewers as a 3D field
dn = lambda n: n / n.mean() - 1.0
k_edges = np.geomspace(2 * np.pi / L3 * 0.999, np.pi * N3 / L3, 16)
kin = lambda kb: np.asarray(grf3.power(kb)) / sig3 ** 2                                  # input of delta / sigma
rows3 = {}
for lab, f, b, c, sty in (("δ_n H I / b", dn(n3[0]), b_lin["HI"], EC["HI"], dict(marker="o", ms=7, lw=2)),
                          ("δ_n Si III / b", dn(n3[1]), b_lin["SiIII"], EC["SiIII"], dict(marker="s", ms=6, lw=2, ls="--"))):
    kb, Pb, eb, nm = binned_power(f[None] / b, L3, k_edges)
    rows3[lab] = (Pb, eb, c, sty)
for lab, F, c, sty in (("δ_F Lyα / b_F", F3[0], EC["HI-LyA"], dict(lw=2.4, ls="-")), ("δ_F Si III / b_F", F3[1], EC["SiIII-1206"], dict(lw=2.0, ls=":"))):
    d = F / F.mean() - 1
    kb2, Pb2, eb2, _ = binned_power(grid3(d)[None], L3, k_edges)                        # the line of sight in cMpc of the box
    bF2 = np.median(Pb2[:5] / rows3["δ_n H I / b"][0][:5])                            # against the same realisation: no cosmic variance
    rows3[lab + f" (b_F² = {bF2:.2g})"] = (Pb2 / bF2, eb2 / bF2, c, sty)
fig, axs = plt.subplots(2, 1, figsize=(15, 11), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(kb, rows3, kin(kb), ref_label="input P(k)", axs=axs)
axs[0].set_ylabel(r"$k P(k)/\pi$ [cMpc$^{-2}$]"); axs[1].set(xlabel=r"$|k|$ [cMpc$^{-1}$]", ylim=(0.05, 3))
add_panel_letters(axs); fig.suptitle("(a) 3D: the overdensities recover the input; the overfluxes are biased and smoothed", fontsize=21)
plt.show()
for lab, (Pb, eb, *_r) in rows3.items():
    r = Pb / kin(kb)
    print(f"{lab:32s} P / input: k < 0.5 cMpc^-1 mean {r[kb < 0.5].mean():.3f}; at the highest k {r[-1]:.3f}")

In [ ]:
# (b) 1D along the skewers (k in s/km through the Hubble flow of the box)
one = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": jnp.asarray(d)}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=jnp.asarray(d))
p1d = lambda d, dx: OverfluxStatistics1D(one(d), dx).compute_p1d()
nmax = N3 // 2
kk = 2 * np.pi * np.arange(1, nmax + 1) / P3v                                          # [s/km]
Pin1, sig1 = (x / sig3 ** 2 * u3 for x in grf_p1d_along_axis(grf3, kk * u3, axis=1, return_sigma=True))
take = lambda st, scale=u3: np.asarray(st.Pk_total)[1:nmax + 1] * scale
err = lambda st, scale=u3: np.asarray(st.Pk_total_err)[1:nmax + 1] * scale
n3l = n3.reshape(2, N3 * N3, N3)                                                       # the 64 x 64 grid lines along y
n_tot = n3l[0] + n3l[1]
b_tot = (b_lin["HI"] * n_bar["HI"] + b_lin["SiIII"] * n_bar["SiIII"]) / (n_bar["HI"] + n_bar["SiIII"])
dens_rows = {}
for lab, n, b, c, sty in (("δ_n H I / b", n3l[0], b_lin["HI"], EC["HI"], dict(marker="o", ms=6, lw=2)),
                          ("δ_n Si III / b", n3l[1], b_lin["SiIII"], EC["SiIII"], dict(marker="s", ms=5, lw=2, ls="--")),
                          ("δ_n (H I + Si III) / b", n_tot, b_tot, TOL["green"], dict(lw=2.4, ls=":"))):
    st_ = p1d(dn(n) / b, L3 / N3)
    dens_rows[lab] = (take(st_), err(st_), c, sty)
flux_rows = {}
for lab, F, c, sty in (("δ_F Lyα", F3[0], EC["HI-LyA"], dict(lw=2.4)), ("δ_F Si III", F3[1], EC["SiIII-1206"], dict(lw=2.0, ls="--")),
                       ("δ_F total", F3.prod(0), FG, dict(lw=1.6, ls=":"))):
    st_ = p1d(F / F.mean() - 1, P3v / N_v3)
    P_, e_ = take(st_, 1.0), err(st_, 1.0)
    bF2 = np.median(P_[:4] / dens_rows["δ_n H I / b"][0][:4])                          # against the same realisation
    flux_rows[f"{lab} / b_F (b_F² = {bF2:.2g})"] = (P_ / bF2, e_ / bF2, c, sty)
fig, axs = plt.subplots(2, 2, figsize=(24, 11), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(kk, dens_rows, Pin1, ref_label="input, projected to 1D", axs=axs[:, 0])
plot_p1d_input_vs_measured(kk, flux_rows, Pin1, ref_label="input, projected to 1D", axs=axs[:, 1])
axs[0, 0].set_title("(b1) the overdensities, at the grid nodes", fontsize=18); axs[0, 1].set_title("(b2) the overfluxes, on the Lyα frame", fontsize=18)
for a in axs[1]:
    a.set_ylim(0.05, 3)
add_panel_letters(axs.ravel()); fig.suptitle("(b) 1D along the skewers: the same signals against the input projected to 1D", fontsize=21)
plt.show()
pull = (dens_rows["δ_n H I / b"][0] - Pin1) / sig1
print(f"delta_n H I / b at the nodes: chi2/dof = {np.mean(pull ** 2):.2f} over {nmax} modes with the exact single-realisation sigma; "
      f"Lya overflux / b_F at k_N: {flux_rows[list(flux_rows)[0]][0][-1] / Pin1[-1]:.2f}")

**(c) The 3D analysis on the grid of skewers.** The 32 × 32 skewers of (a) form a regular transverse grid
(row-major, N_sk = N_x N_y), so `OverfluxStatistics3D` measures P3D(k_⊥, k_∥) and ξ3D(r_⊥, π) of the Lyα overflux
directly (transverse axes in cMpc, the line of sight in km/s converted to cMpc through the Hubble flow of the box).
In bins of k_⊥, P3D(k_⊥, k_∥)/b_F² follows the input P_in(√(k_⊥² + k_∥²)) at low k_∥ and falls at high k_∥, where the
thermal widths smooth the flux along the line of sight only: that is the anisotropy the isotropic average of (a)
hides.

In [ ]:
from skewerjax.utils.plotting import tol_cmap
dF3 = F3[0] / F3[0].mean() - 1
dpar = L3 / N_v3
st3 = OverfluxStatistics3D(DeltaFieldBuilder(jnp.asarray(tau3[:1])).build(), dpar, 2 * L3 / N3).compute_xi3d()
Pb, kperp, _, kpar, _ = st3.bin_p3d_kperp(5)
Pb, kperp, kpar = np.asarray(Pb), np.asarray(kperp), np.asarray(kpar)
kbF, PbF, _, _ = binned_power(grid3(dF3)[None], L3, k_edges)
bF2_3d = np.median(PbF[:5] / rows3["δ_n H I / b"][0][:5])
cm = tol_cmap("iridescent", reverse=True)
fig, axs = plt.subplots(2, 2, figsize=(24, 11), layout="constrained", gridspec_kw={"height_ratios": [2.2, 1.3], "width_ratios": [1.2, 1]})
for i in range(len(kperp)):
    c = cm(0.25 + 0.7 * i / max(len(kperp) - 1, 1))
    exp_ = kin(np.sqrt(kperp[i] ** 2 + kpar[1:] ** 2))
    axs[0, 0].loglog(kpar[1:], Pb[i, 1:] / bF2_3d, color=c, lw=2, label=rf"$k_\perp$ = {kperp[i]:.2f}")
    axs[0, 0].loglog(kpar[1:], exp_, color=c, **MS["reference"])
    axs[1, 0].semilogx(kpar[1:], Pb[i, 1:] / bF2_3d / exp_, color=c, lw=2)
axs[1, 0].axhline(1, color=FG, ls="--", lw=1.2); axs[1, 0].set_ylim(0, 2)
axs[0, 0].set(ylabel=r"$P_{\rm 3D}/b_F^2$ [cMpc$^3$]", title=r"P3D of the Lyα overflux in bins of $k_\perp$ [cMpc$^{-1}$] (thick, light: input)")
axs[1, 0].set(xlabel=r"$k_\parallel$ [cMpc$^{-1}$]", ylabel="measured / input"); axs[0, 0].legend(fontsize=15, ncol=2)
xb, rperp, _, pi, _ = st3.bin_xi3d_rperp(12)
xb = np.asarray(xb); lim = float(np.nanmax(np.abs(xb)))
im = axs[0, 1].pcolormesh(np.asarray(pi), np.asarray(rperp), xb, cmap=tol_cmap("BuRd"), vmin=-lim, vmax=lim, shading="auto")
fig.colorbar(im, ax=axs[0, 1], label=r"$\xi_{\rm 3D}$"); axs[0, 1].set(xlabel=r"$\pi$ [cMpc]", ylabel=r"$r_\perp$ [cMpc]", title=r"$\xi_{\rm 3D}(r_\perp, \pi)$ of the Lyα overflux")
mid = np.argmin(np.abs(np.asarray(pi)))
axs[1, 1].plot(np.asarray(rperp), xb[:, mid], color=EC["HI-LyA"], lw=2, label=r"$\pi \approx 0$ (across)")
axs[1, 1].plot(np.abs(np.asarray(pi)), xb[0], color=TOL["yellow"], lw=2, ls="--", label=r"$r_\perp \approx 0$ (along)")
axs[1, 1].axhline(0, color=FG, lw=0.8); axs[1, 1].set(xlabel="separation [cMpc]", ylabel=r"$\xi_{\rm 3D}$"); axs[1, 1].legend(fontsize=15)
add_panel_letters(axs.ravel()); fig.suptitle("(c) The 3D statistics of the Lyα overflux on the 32 × 32 skewers", fontsize=21); plt.show()
print(f"b_F^2 from the isotropic P(k) of (a): {bF2_3d:.3g}; the line of sight is smoothed: P3D / input at the highest k_par, lowest k_perp bin "
      f"= {Pb[0, -1] / bF2_3d / kin(np.sqrt(kperp[0] ** 2 + kpar[-1] ** 2)):.2f}")

All the statistics kernels (`p1d_kernel`, `p3d_kernel`, `xi_from_power`, `binned_mean`) are jitted; recovery and
exactness checks are in `scripts/validation/check_summary_statistics.py`.